# 05 - Why the Leaderboard Disagreed

Notebook 04 ended with a hold-out score of **0.15171** and a forecast, recorded before
submitting, that the leaderboard would land near **0.143**.

It came in at **0.128**.

This notebook is the investigation that followed. It is not a tidy narrative arriving at
an answer, because that is not what happened: five hypotheses were stated and tested,
three were falsified, one was confirmed, and one of the analyses turned out to be built
wrong in a way that made the story *worse* once corrected.

What it establishes, in order:

| # | Question | Answer |
|---|---|---|
| 1 | Do the high-drift features explain it? | No - falsified at two thresholds |
| 2 | Was the forecast even computed correctly? | No - cosine weights subgroups by magnitude, not count |
| 3 | Is the test set a different kind of data? | No - it is a continuation of training |
| 4 | Was the hold-out an unlucky read? | **Yes - a lucky one, and it explains 46%** |
| 5 | Would tuning have helped? | No - the gain is selection noise |
| 6 | Was the submitted model even the right one? | **No - it was handicapped twice** |

The through-line is that almost everything learned here is about **measurement** rather
than modelling - and that on this problem, effects small enough to need careful
measurement are usually too small to matter.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import sys
sys.path.insert(0, "..")
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config

plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True,
                     "grid.alpha": .3, "font.size": 10})
pd.set_option("display.width", 140)

feat = Path(load_config().paths.features)
print("results directory:", feat)

results directory: C:\mscapital_data\features


---

## The prediction was wrong

The submission was made on 2026-09-03. **Leaderboard: 0.128.**

The prediction above was `0.143`, with the explicit caveat that "a score materially BELOW
~0.143 would need an explanation beyond the spread regime". 0.128 is materially below it.
So the caveat is now due.

In [2]:
holdout_bucketed = 0.14891      # printed above
predicted        = 0.14345      # same model, re-weighted to the test spread mix
actual           = 0.128        # Kaggle leaderboard, 2026-09-03

total   = actual - holdout_bucketed
spread  = predicted - holdout_bucketed
residual = actual - predicted

print(f"hold-out (bucket-weighted)   {holdout_bucketed:+.5f}")
print(f"predicted                    {predicted:+.5f}   ({spread:+.5f} from spread mix)")
print(f"actual                       {actual:+.5f}   ({residual:+.5f} unexplained)")
print()
print(f"total degradation            {total:+.5f}  ({total/holdout_bucketed*100:+.1f}%)")
print(f"  explained by spread mix    {spread/total*100:5.1f}%")
print(f"  NOT explained              {residual/total*100:5.1f}%")

hold-out (bucket-weighted)   +0.14891
predicted                    +0.14345   (-0.00546 from spread mix)
actual                       +0.12800   (-0.01545 unexplained)

total degradation            -0.02091  (-14.0%)
  explained by spread mix     26.1%
  NOT explained               73.9%


### What that means

The mechanism was real but small. Spread-regime mix accounts for roughly a **quarter** of
the degradation; three quarters came from something the analysis did not model. The
direction was right and the magnitude was wrong by about a factor of three.

That is a more useful result than a correct prediction would have been, because it says
something specific: **the hold-out is not a good proxy for the leaderboard here, and the
reason is not the one variable I could measure.** Months 65-70 are the tail of the same
training distribution; the test set is a different period entirely, and notebook 03 already
showed the rate features shifting hardest. The natural next experiment is to retrain
without the highest-drift features and see whether a small hold-out cost buys a larger
leaderboard gain — but that costs another submission to verify, so it is stated as a
hypothesis rather than claimed as a result.

### Why this is left in

The tempting edit is to delete the failed prediction, or to soften it into something that
survives contact with the number. Both would destroy the only thing that makes it worth
anything. A prediction is only evidence of understanding if it was recorded **before** the
answer arrived and is reported honestly **after**. The wrong number stays where it was
written, with the correction underneath it.

The transferable lesson is about the estimator, not this model: a hold-out carved from the
end of the training period measures generalisation across *time within one regime*, not
across regimes. It will read optimistically whenever the deployment distribution differs
from the training distribution in ways the hold-out cannot see — which, in production
finance, is most of the time.

---

## Testing the suspect: does drift actually predict degradation?

Notebook 03 nominated the high-drift rate features as the likely source of the
unexplained 74%. Nominating a suspect is not evidence.

The obvious test — retrain without them and submit — costs a submission and returns one
bit. The mechanism can be tested on the training data alone instead. If high-drift
features are a liability **under distribution shift**, then dropping them should help
*more* when the gap between training and evaluation is *larger*. The training set spans
71 months, so that gap can be dialled directly:

```
train on months 0-34, FIXED          <- never changes, so only the gap varies
evaluate on 36-40, 42-46, ... 66-70  <- increasing distance into the future
```

The quantity of interest is neither score but their difference,
`lift(gap) = cosine(pruned) - cosine(full)`. Both feature sets see identical rows, folds
and seeds, so run-to-run noise is shared and cancels in the difference — which matters,
because the effect being looked for is about the size of the fold-to-fold noise. No early
stopping: stopping on the evaluation block would tune to the thing being measured.

A drift mechanism predicts lift rises with gap. A flat line falsifies it.

In [3]:
rob = pd.read_csv(feat / "drift_robustness_t020.csv")

print(rob[["block", "gap_months", "n", "full", "pruned", "lift"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.5f}"))

g = rob.gap_months.to_numpy(float)
slope, r = np.polyfit(g, rob.lift, 1)[0], np.corrcoef(g, rob.lift)[0, 1]
resid = rob.lift - np.polyval(np.polyfit(g, rob.lift, 1), g)
se = np.sqrt((resid**2).sum() / (len(g) - 2) / ((g - g.mean())**2).sum())

print()
print(f"lift vs gap : slope {slope:+.6f}/month, Pearson r {r:+.3f}")
print(f"95% CI over the full 32-month span: "
      f"[{(slope-1.96*se)*32:+.5f}, {(slope+1.96*se)*32:+.5f}] cosine")
print(f"unexplained leaderboard gap       :  {0.01545:+.5f} cosine")

block  gap_months     n    full  pruned     lift
36-40           2 88685 0.12954 0.12815 -0.00139
42-46           8 88898 0.12546 0.12791  0.00245
48-52          14 89096 0.12935 0.13259  0.00324
54-58          20 88573 0.12736 0.12957  0.00221
60-64          26 88704 0.12810 0.12842  0.00032
66-70          32 87079 0.13856 0.13789 -0.00066

lift vs gap : slope -0.000018/month, Pearson r -0.109
95% CI over the full 32-month span: [-0.00578, +0.00462] cosine
unexplained leaderboard gap       :  +0.01545 cosine


**Falsified.** The lift does not rise with the gap — it is positive in the middle and
negative at both ends, with a correlation indistinguishable from zero. Even the optimistic
edge of the confidence interval is several times too small to account for the gap the
experiment was built to explain.

A null result invites the objection that the knob was set wrong, so the same experiment was
re-run pruning far more aggressively.

In [4]:
rob10 = pd.read_csv(feat / "drift_robustness_t010.csv")

for name, r_ in (("|shift| >= 0.20  (57 dropped)", rob), ("|shift| >= 0.10  (93 dropped)", rob10)):
    sl = np.polyfit(g, r_.lift, 1)[0]
    print(f"{name}: slope {sl:+.6f}/month  r {np.corrcoef(g, r_.lift)[0,1]:+.3f}  "
          f"mean lift {r_.lift.mean():+.5f}")

# The full arm is identical in both runs - same seeds, same columns. An unplanned
# control: it confirms only the pruned arm changed.
print()
print("full arm reproduced exactly across the two runs:",
      bool((rob.full.round(10) == rob10.full.round(10)).all()))

|shift| >= 0.20  (57 dropped): slope -0.000018/month  r -0.109  mean lift +0.00103
|shift| >= 0.10  (93 dropped): slope -0.000086/month  r -0.352  mean lift -0.00080

full arm reproduced exactly across the two runs: True


Pruning harder does not rescue the hypothesis — it buries it. At the aggressive threshold
the trend is *negative* (pruning helps **less** at longer gaps, the opposite of the
prediction) and the mean lift is negative too: dropping the high-drift features costs
skill on average rather than buying robustness.

The `full` arm reproducing to ten decimal places across the two runs is an unplanned
control worth keeping: it confirms the two arms differed only in their column list, so the
lift really does isolate the pruning.

But the more interesting number was in the `full` column all along, and it was not what
the experiment was looking for.

In [5]:
for col in ("full", "pruned"):
    s = np.polyfit(g, rob[col], 1)[0]
    print(f"{col:7s} vs gap: slope {s:+.6f}/month  "
          f"r {np.corrcoef(g, rob[col])[0,1]:+.3f}   "
          f"range {rob[col].min():.5f} - {rob[col].max():.5f}")

best = rob.loc[rob.full.idxmax()]
print()
print(f"best block is {best.block} - the FURTHEST one, {best.gap_months:.0f} months out")

full    vs gap: slope +0.000243/month  r +0.596   range 0.12546 - 0.13856
pruned  vs gap: slope +0.000225/month  r +0.647   range 0.12791 - 0.13789

best block is 66-70 - the FURTHEST one, 32 months out


### The framing was wrong, not just the suspect

Within the training period, model skill **does not decay with elapsed time at all**. The
slope is slightly *positive*, and the most distant block scores highest. Train on months
0-34 and evaluate 32 months later, and the model does no worse than it does two months
later.

That undercuts the whole explanation, not just the rate-feature version of it. "The market
regime drifts over time, so a later test set scores worse" makes a prediction about the
training period too — and the training period says no. Whatever separates the test set
from the hold-out, it is not that the test set is *later*.

So the honest state of the question:

| Hypothesis | Status | Explains |
|---|---|---|
| Spread-regime mix shift | measured | 26% |
| High-drift rate features | **falsified** at two thresholds | not detectable; pruning is mildly harmful |
| Elapsed time / regime drift | **falsified** | no decay within 71 months |
| — | remaining | **74%, unaccounted for** |

Candidates that remain untested, in the order I would try them: the test set differs
categorically rather than temporally (its order rate is 36% higher than training — a large
difference that does not appear anywhere inside the training span); cosine is a *pooled*
magnitude-weighted metric, so a different volatility mix changes the score even at
identical per-sample skill; and design choices were made against CV folds neighbouring the
hold-out, which buys some selection optimism that a genuinely independent test set would
not honour.

Two hypotheses stated in advance, two tested, two rejected. The gap is still mostly
unexplained — and that is the accurate thing to report.

---

## The forecast was also built wrong

Re-reading the prediction after it failed turned up a flaw in the *method*, independent of
the result. Cosine is computed over the pooled vector, without centring, so splitting the
samples into disjoint groups factors it exactly:

$$\cos(y, p) \;=\; \sum_g \cos_g \cdot w_g,
\qquad w_g = rac{\lVert y_gVert \, \lVert p_gVert}{\lVert yVert \, \lVert pVert}$$

A pooled cosine is a weighted average of subgroup cosines — but the weights are products
of **magnitudes**, not sample counts. (They sum to at most 1 by Cauchy-Schwarz, with
equality only when every group scores alike.)

The forecast reweighted the per-quartile scores by the test set's *sample shares*. That
assumes the magnitude weights track the counts. They do not.

In [6]:
from src.evaluation.cosine_decomposition import decompose, verify_identity

dec = pd.read_csv(feat / "cosine_decomposition.csv")
print(dec[["group", "n", "cosine", "y_rms", "count_weight", "weight"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.5f}"))

meta = json.loads((feat / "cosine_decomposition_meta.json").read_text())
print()
print(f"identity check: pooled {meta['pooled']:.10f} = rebuilt {meta['rebuilt']:.10f}"
      f"  (error {meta['abs_error']:.1e})")

        group     n  cosine   y_rms  count_weight  weight
Q1 (tightest) 25915 0.13148 0.00351       0.24735 0.24394
           Q2 25918 0.13524 0.00338       0.24738 0.22074
           Q3 25892 0.13477 0.00328       0.24713 0.20870
  Q4 (widest) 25909 0.19415 0.00350       0.24729 0.31238
          NaN  1136 0.20836 0.00151       0.01084 0.00486

identity check: pooled 0.1517128275 = rebuilt 0.1517128275  (error 2.8e-17)


The quartiles are equal-sized by construction, so every count weight is 0.247 — but the
weights the metric applies run from **0.209 to 0.312**, a 50% spread. And `y_rms` barely
moves across buckets (0.0033-0.0035), so almost none of that comes from the targets: it is
the *predictions* that are larger in wide-spread samples, which is what a volatility-aware
model should do.

Crucially, the heaviest weight lands on **Q4 — the bucket where the model is strongest**.

So what does the forecast look like when redone with the right weights? On the test set
$\lVert p_gVert$ is directly observable (the submitted predictions are in hand); only
$\lVert y_gVert$ is not, and it is carried over from the hold-out as a per-bucket target
RMS — a far weaker assumption than count-weighting, given that RMS varies by 7% while the
weights vary by 50%.

In [7]:
fc = pd.read_csv(feat / "cosine_forecast_corrected.csv")
print(fc[["group", "n", "holdout_cosine", "count_weight", "weight"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.5f}"))

m = json.loads((feat / "cosine_decomposition_meta.json").read_text())
pooled, actual = m["pooled"], m["actual_leaderboard"]
corrected = m["forecast_magnitude_weighted"]

print()
print(f"hold-out, as measured        {pooled:+.5f}")
print(f"forecast, count-weighted     {m['forecast_count_weighted']:+.5f}   <- the method used")
print(f"forecast, magnitude-weighted {corrected:+.5f}   <- corrected")
print(f"actual                       {actual:+.5f}")
print()
print(f"spread mix now explains {(pooled-corrected)/(pooled-actual)*100:.0f}% of the gap"
      f"  (was claimed: 26%)")

        group      n  holdout_cosine  count_weight  weight
Q1 (tightest) 223201         0.13148       0.34693 0.34493
           Q2 162363         0.13524       0.25237 0.22056
           Q3 138631         0.13477       0.21548 0.18697
  Q4 (widest) 119162         0.19415       0.18522 0.24755

hold-out, as measured        +0.15171
forecast, count-weighted     +0.14474   <- the method used
forecast, magnitude-weighted +0.14844   <- corrected
actual                       +0.12800

spread mix now explains 14% of the gap  (was claimed: 26%)


### The correction makes the story worse, which is why it is worth making

Fixing the weighting moves the forecast **away** from the outcome, not towards it. Because
cosine over-weights Q4, where the model is strongest, it partly cancels the penalty from
Q1's larger share — so the spread mechanism was weaker than the original analysis claimed,
not stronger. Its share of the gap drops from 26% to about **14%**.

| Hypothesis | Status | Explains |
|---|---|---|
| Spread-regime mix shift | measured, then **corrected downward** | ~14% |
| High-drift rate features | falsified at two thresholds | not detectable |
| Elapsed time / regime drift | falsified | no decay over 71 months |
| — | remaining | **~86%, unaccounted for** |

There is a temptation, on finding an error in your own analysis, to look for the version of
the fix that rescues the conclusion. The fix here does the opposite, and reporting it that
way is the whole point: the error was in the method, so it had to be corrected regardless
of which direction the answer moved.

### The general lesson

Any subgroup analysis of a cosine score — error slices, fairness-style breakdowns,
"where is my model weak" tables — must weight groups by magnitude, not by row count, or it
describes a metric nobody is being scored on. The failure mode is not subtle:
`tests/test_cosine_decomposition.py` builds a model that is near-perfect on the
small-magnitude half of the data and useless on the large-magnitude half. Counting rows
calls it skilful at +0.48. The metric scores it **−0.04** — no skill at all.

And there is a second, harder consequence. The corrected weights need
$\lVert y_gVert$ on the test set, which is unobservable. So a subgroup-reweighting
forecast of a cosine score is **not fully computable in advance** — it always rests on an
assumption about unobserved target magnitudes. Count-weighting is one such assumption, and
a poor one. That, rather than any particular number, is the durable finding here.

---

## Is the test set *later*, or *different*?

Three explanations are now gone: the spread mix is small, drift-pruning does nothing, and
skill does not decay with elapsed time. That leaves the possibility that the test set is
not a continuation of the training period at all.

That is measurable. Train a classifier to tell train rows from test rows; the AUC says how
distinguishable they are. On its own the number means nothing — any two separated periods
are somewhat distinguishable — so it needs a scale. The same measurement is repeated
*inside* the training data at growing temporal distance, tracing how distinguishability
grows with elapsed time in this market, for these features.

In [8]:
adv = pd.read_csv(feat / "adversarial_auc.csv")
print(adv[["comparison", "kind", "distance_months", "auc"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

               comparison          kind  distance_months    auc
             months 10-19  within-train          10.0000 0.7543
             months 20-29  within-train          20.0000 0.8083
             months 30-39  within-train          30.0000 0.8835
             months 40-49  within-train          40.0000 0.9075
             months 50-59  within-train          50.0000 0.9026
             months 60-70  within-train          60.5000 0.8812
TEST (vs reference block) train-vs-test              NaN 0.9182
     TEST (vs last block) train-vs-test              NaN 0.7490
   TEST (vs pooled train)    diagnostic              NaN 0.7910


**The test set is a continuation, not a departure.** Against the most recent training
months it scores **0.749** — *lower* than months 10-19 against months 0-9 (0.754). Two
adjacent blocks inside the training data are more distinguishable from each other than the
test set is from the end of training. Whatever is costing 0.024, it is not that the test
set comes from somewhere else.

### A bug worth showing

The first version of this compared *pooled* training data against the test set and got
0.791, which read as "less shifted than most training blocks" — a tidy answer, and wrong.
Pooling all 71 months makes that side far more heterogeneous, and a narrow block is
naturally hard to separate from a broad mixture that contains something like it. The AUC
was depressed for a reason with nothing to do with the test set.

The row is kept in the table as a diagnostic. Comparisons are only comparable when both
sides are built the same way, and `tests/test_adversarial.py` pins that down with a case
where pooling one side provably lowers the AUC.

---

## The estimator, not the model

Everything above looked for something wrong with the *test set*. Nothing was. So the
remaining suspect is the thing that produced the number in the first place.

The hold-out is a single contiguous stretch — months 65-70 — chosen for being **last**,
not for being **typical**. If those months happen to be favourable, the hold-out overstates
what a fresh period yields, and no amount of care elsewhere would reveal it: the score is
measured perfectly, on a sample of one period.

So: hold the model fixed (trained once on months 0-34) and score it on every later period
in turn. Training set, features, seeds and rounds are identical, so the only thing varying
is which months are being predicted. Any spread is period difficulty.

In [9]:
per = pd.read_csv(feat / "period_difficulty.csv")
pm = json.loads((feat / "period_difficulty_meta.json").read_text())

print(per[["block", "n", "cosine"]].to_string(index=False,
                                              float_format=lambda v: f"{v:,.5f}"))
print()
print(f"hold-out months 65-70   {pm['holdout_cosine']:+.5f}")
print(f"median of other blocks  {pm['typical_cosine']:+.5f}")
print(f"the hold-out is {(pm['ratio']-1)*100:.1f}% above a typical period, "
      f"above {pm['holdout_percentile']*100:.0f}% of all blocks")

block     n  cosine
36-38 53130 0.12025
39-41 53344 0.13647
42-44 53251 0.12202
45-47 53453 0.13087
48-50 53452 0.13451
51-53 53412 0.12634
54-56 53229 0.12493
57-59 53131 0.13912
60-62 53220 0.12370
63-65 53175 0.13680
66-68 52028 0.14761
69-70 35051 0.11712

hold-out months 65-70   +0.13854
median of other blocks  +0.12861
the hold-out is 7.7% above a typical period, above 83% of all blocks


Period difficulty swings from **0.117 to 0.148** — a range of 26% — with a fixed model on
fixed features. The hold-out sits at the **83rd percentile**. It was a good draw.

In [10]:
wf = pd.read_csv(feat / "walkforward_summary.csv")
cv_mean = float(wf.loc[wf.model == "ensemble", "cosine_mean"].iloc[0])

print(f"reported hold-out score        {pm['reported']:+.5f}")
print(f"de-biased to a typical period  {pm['debiased']:+.5f}")
print(f"walk-forward CV mean           {cv_mean:+.5f}   <- independent route")
print(f"actual leaderboard             {pm['actual']:+.5f}")
print()
print(f"agreement between the two routes: {abs(pm['debiased']-cv_mean):.5f}")
print(f"period luck explains {pm['share_of_gap']*100:.0f}% of the gap")

reported hold-out score        +0.15171
de-biased to a typical period  +0.14084
walk-forward CV mean           +0.14088   <- independent route
actual leaderboard             +0.12800

agreement between the two routes: 0.00004
period luck explains 46% of the gap


### The two routes agree to four decimal places

De-biasing the hold-out by its measured period difficulty gives **0.14084**. Averaging the
walk-forward folds across five *different* periods gives **0.14088**. Nothing was tuned to
make those meet.

That is the finding, and it is uncomfortable: **the CV mean was the better estimate all
along, and I led with the hold-out instead.** The hold-out was chosen on the strongest
methodological grounds available — untouched by feature design, model selection, tuning or
early stopping, measured exactly once. All of that is true and none of it helps, because
purity was never the binding constraint. Period difficulty was, and against that a single
contiguous stretch is a sample of size one, however cleanly it is handled.

The walk-forward mean is *less* pure — its folds informed decisions — yet it averages over
five periods and lands within 0.00004 of the de-biased answer.

| | estimate | error vs leaderboard |
|---|---:|---:|
| Hold-out, months 65-70 | +0.15171 | +0.02371 |
| Walk-forward CV mean | +0.14088 | +0.01288 |
| Leaderboard | +0.12800 | — |

**Where the gap stands**

| Hypothesis | Verdict | Share |
|---|---|---:|
| The hold-out was a lucky period | **confirmed**, two independent routes | **46%** |
| Spread-regime mix shift | real, after correcting the weighting | ~14% |
| High-drift rate features | falsified at two thresholds | — |
| Skill decays with elapsed time | falsified | — |
| Test set is categorically different | falsified — it is a continuation | — |

The two surviving effects should not simply be added: both are ways of saying the hold-out
period was atypical, and they may be measuring overlapping parts of the same thing. Taken
together they account for somewhere between **46% and 60%** of a gap that began as fully
unexplained.

### Is the rest just noise?

Period difficulty varies a lot, so the obvious next question is whether the leftover gap
is simply the test set's own unlucky draw. It is not, and the same table answers it.

In [11]:
sd = per.cosine.std(ddof=1)
n_blocks = 647_896 / per.n.mean()          # test set, in units of these blocks
noise = sd / np.sqrt(n_blocks)
residual = pm["debiased"] - pm["actual"]

print(f"block-to-block difficulty     std {sd:.5f}")
print(f"test spans ~{n_blocks:.0f} such blocks -> its own period noise ~{noise:.5f}")
print(f"residual gap after de-biasing     {residual:+.5f}  ({residual/noise:.1f} sigma)")

block-to-block difficulty     std 0.00908
test spans ~13 such blocks -> its own period noise ~0.00256
residual gap after de-biasing     +0.01284  (5.0 sigma)


The test set averages over roughly thirteen blocks, so its own period draw is worth about
**±0.0026** — while the residual is **0.0128**, some five standard deviations out. The
remainder is a real effect, not sampling noise, and it is still unidentified.

### What I would do differently

Report the walk-forward mean as the headline and the hold-out as a *check* on it, rather
than the reverse. A held-out period answers "did I leak?" It does not answer "what will
this score next period", and those were quietly treated as the same question.

---

## Hyperparameter search: how much of a gain is real?

The plan deprioritised tuning on the grounds that it buys little per hour of work. That
was a judgement, not a measurement, so here is the measurement.

The search is routine — TPE over eight LightGBM parameters, walk-forward cosine as the
objective. What matters is the accounting around it, because by now the noise floor is
known and it is **high**: fold-to-fold std 0.0041, block-to-block period std 0.0091.
Running N trials and keeping the best does not only find good parameters, it also finds
favourable noise, and at N in the dozens that term is the same size as the effect being
chased.

So the winner is re-scored under a fresh resampling of the identical protocol — new
bagging and feature-sampling seeds, nothing else changed. What survives belongs to the
parameters; what evaporates was the search fitting noise.

In [12]:
tun = json.loads((feat / "tuning_result.json").read_text())

print(f"trials completed          {tun['trials_completed']} of {tun['n_trials']}"
      f"  (stopped by the {tun['time_budget_min']:.0f}-min budget)")
print(f"search set                {tun['sample_frac']:.0%} of train, "
      f"{tun['search_folds']} folds")
print()
print(f"baseline CV               {tun['baseline_cv']:+.5f}")
print(f"best trial CV             {tun['best_cv']:+.5f}")
print(f"gain the search claims    {tun['claimed_gain']:+.5f}")
print(f"gain surviving new seeds  {tun['honest_gain']:+.5f}")
print(f"SELECTION OPTIMISM        {tun['selection_optimism']:+.5f}"
      f"   ({tun['selection_optimism']/tun['claimed_gain']:.0%} of the claim)")

trials completed          23 of 40  (stopped by the 60-min budget)
search set                30% of train, 3 folds

baseline CV               +0.13691
best trial CV             +0.13951
gain the search claims    +0.00260
gain surviving new seeds  +0.00062
SELECTION OPTIMISM        +0.00198   (76% of the claim)


**Three quarters of the gain was noise.** What is left, +0.0006, is a seventh of the
fold-to-fold std and a fifteenth of the period std.

One objection remains: the search ran on 30% of the rows, and several of these knobs scale
with sample size — `min_data_in_leaf` came back at 1018, tuned against 377k rows rather
than 1.26M. So both arms were re-run on the full dataset, paired: identical folds,
identical rows, identical seeds, only the parameters differing.

In [13]:
con = json.loads((feat / "tuning_confirm.json").read_text())

print(f"full data, {con['full_data_rows']:,} rows, last {con['n_folds']} folds, paired")
print(f"  tuned     {con['tuned_mean']:+.5f}   {[f'{x:+.5f}' for x in con['tuned_folds']]}")
print(f"  baseline  {con['baseline_mean']:+.5f}   {[f'{x:+.5f}' for x in con['baseline_folds']]}")
print(f"  paired difference {con['paired_gain']:+.5f}")
print(f"  per fold          {[f'{d:+.5f}' for d in con['per_fold_diff']]}")
print(f"  folds improved    {con['folds_improved']} of {con['n_folds']}")

full data, 1,257,637 rows, last 3 folds, paired
  tuned     +0.14143   ['+0.13899', '+0.14339', '+0.14191']
  baseline  +0.14181   ['+0.14034', '+0.14187', '+0.14322']
  paired difference -0.00038
  per fold          ['-0.00135', '+0.00152', '-0.00131']
  folds improved    1 of 3


### It does not transfer. It reverses.

On full data the tuned configuration is **worse** than the hand-chosen defaults, by
−0.00038, and only one fold of three improves. The per-fold differences alternate sign
(−0.00135, +0.00152, −0.00131) — the signature of noise, not of a parameter effect.

| | gain |
|---|---:|
| Claimed by the search | +0.00260 |
| Surviving fresh seeds, 30% data | +0.00062 |
| **Full data, paired** | **−0.00038** |

So the honest answer is that **tuning bought nothing**. The defaults — chosen by hand, for
stated reasons about 1.26M rows and memory — are as good as a 23-trial TPE search, and the
apparent improvement was selection noise that did not survive either check.

That is a useful result rather than a wasted hour. "I skipped tuning" is a gap in a
portfolio; "I ran the search, and here is the measurement showing it buys 0.0006 before
transfer and nothing after" is a finding. The original decision to deprioritise it now
rests on evidence instead of judgement.

### The reusable part

The apparatus generalises past this dataset. Any best-of-N result on a noisy objective is
inflated by the maximum of N noise draws, and the fix costs one extra evaluation: re-score
the winner under a fresh resampling of the same protocol and report both numbers. Had only
the first number been reported here, this notebook would be claiming a +0.0026 improvement
that reverses sign on the full data.

### A cost note

The first attempt at this search was allowed to reach `num_leaves` 511, `max_bin` 255 and
`learning_rate` 0.01. The last of those is the trap: at a low enough rate early stopping
never fires, so every trial runs all of its rounds. Trials in that corner cost roughly ten
times the baseline, and a search budgeted at an hour was killed after three with no result.

A search space is a compute budget written in another notation. The version here is bounded
so the worst trial costs about twice the baseline, carries a wall-clock cap that ends the
run regardless — it stopped at 23 of 40 trials and said so — and logs every trial as it
lands, because a long job you cannot observe is indistinguishable from a hung one.

---

## The submitted model was handicapped

Everything so far hunted for something wrong with the data, the metric or the estimator.
The last place to look was the artefact itself — and its own metadata gives it away.

| | submitted | what the project's own CV recommends |
|---|---|---|
| model | single LightGBM | ensemble, ahead in **5 folds of 5** by +0.0022 |
| training months | 0-63 (**64** of 71) | as many as possible |

The second is the more interesting mistake. `finalize.py` keeps months 65-70 out so the
hold-out can be read once — correct for *measuring*. But a hold-out has done its job the
moment it is read, and carrying it through to the shipped artefact throws away 7 months.
The protocol for measuring was silently reused as the protocol for shipping.

### First, what is the discarded data actually worth?

Measurable without a submission: fix the evaluation block at months 65-70 and vary where
training stops. The confound is that extending the window adds **volume** and **recency**
at once, so a second arm holds the training span constant at 52 months and moves only the
endpoint.

In [14]:
rec = pd.read_csv(feat / "recency.csv")
print(rec[["arm", "window", "months", "gap_to_eval", "cosine", "seed_std"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.5f}"))

rm = json.loads((feat / "recency_meta.json").read_text())
print()
print(f"cumulative   slope {rm['cumulative_slope_per_month']:+.6f} cosine per month of staleness")
print(f"fixed window slope {rm['fixed_window_slope_per_month']:+.6f}")

         arm window  months  gap_to_eval  cosine  seed_std
  cumulative   0-51      52           14 0.14517   0.00036
  cumulative   0-57      58            8 0.14503   0.00154
  cumulative   0-63      64            2 0.15273   0.00011
fixed_window   0-51      52           14 0.14517   0.00036
fixed_window   6-57      52            8 0.14198   0.00150
fixed_window  12-63      52            2 0.14307   0.00101

cumulative   slope -0.000630 cosine per month of staleness
fixed window slope +0.000175


**It is volume, not recency.** Hold the training span at 52 months and move it 12 months
closer to the evaluation block: nothing happens (0.14517 -> 0.14307, the wrong way, inside
seed noise). Let the window *grow* instead and the score rises by +0.0076. The extra months
help because they are extra data, not because they are recent.

That agrees with the earlier finding that skill does not decay with elapsed time. Two
independent experiments now say the same thing: in this market the feature-target
relationship does not go stale.

### The prediction

Ensemble +0.0022 from CV, plus roughly +0.0025 for four more months of data at the
measured rate of +0.00063/month. Call it **+0.0047**, so a leaderboard around
**0.132-0.133**.

In [15]:
print(f"{'previous submission':32s} 0.128   single LightGBM, months 0-63")
print(f"{'predicted':32s} 0.132-0.133")
print(f"{'actual':32s} 0.129   ensemble, months 0-67")
print()
print(f"{'predicted gain':32s} +0.0047")
print(f"{'realised gain':32s} +0.0010   ({0.0010/0.0047:.0%} of it)")

previous submission              0.128   single LightGBM, months 0-63
predicted                        0.132-0.133
actual                           0.129   ensemble, months 0-67

predicted gain                   +0.0047
realised gain                    +0.0010   (21% of it)


### Right direction, wrong size — for the third time

The model did improve, and it improved in the predicted direction. It improved by about a
fifth of the predicted amount.

That is now a **pattern**, and it is the most useful thing in this notebook:

| Prediction | Predicted | Actual |
|---|---:|---:|
| Leaderboard from the hold-out | 0.143 | 0.128 |
| Spread-mix share of the gap | 26% | ~14% |
| Gain from ensemble + more data | +0.0047 | +0.0010 |

Three forecasts, all built from carefully measured internal quantities, all overshooting
in the same direction. The individual explanations differ, but the common cause does not:
**effects of order 0.002-0.005, measured on internal splits, are near the resolution limit
of this problem.** Fold-to-fold std is 0.0041 and period-to-period std is 0.0091. A CV
difference of +0.0022 that appears in 5 folds of 5 is a real ordering of the models — and
still buys almost nothing externally, because what separates them is small next to what
separates one period from another.

The practical rule this leaves: on a problem with this noise structure, treat any internal
gain below roughly the fold-to-fold std as directional evidence about *which* model to
prefer, never as a quantity that will show up on a leaderboard. Both submissions here are
consistent with that rule; the forecasts were not.

### What was fixed, and what it cost

The shipped artefact now trains on 68 of 71 months instead of 64, blends three models
instead of one, and — having no untouched period left — deliberately reports **no** hold-out
score. Quoting finalize.py's 0.15171 beside a differently-trained model would be a category
error. Its honest internal estimate is the walk-forward mean; its test was the leaderboard.

One bug is worth recording. The first build pulled the inner estimators out of their
wrappers to store them, which silently discarded `RidgeModel`'s median imputation — and the
feature layer emits NaN by design, so prediction died on the first short-window row. The
wrappers already share one `predict` contract; unwrapping them threw away the abstraction
that existed precisely to prevent this. `tests/test_ship.py` reproduces both halves.

And the blend window is two months, not one. Fitted on month 70 alone, NNLS handed xgboost
a weight of **0.71** — against a CV that finds it and LightGBM indistinguishable. The base
models are highly correlated, so their differences are mostly noise and the split swings on
very little. Two months gives 0.63 / 0.32 / 0.05, which matches the CV ordering.